## 16. Credit-Stress ECL: Equation-by-Equation Derivation

[![content: AI-generated](../_static/badges/ai-generated.svg)](ai-acknowledgement) [![content: human-reviewed & edited](../_static/badges/human-reviewed-edited.svg)](ai-acknowledgement)

[4.7 Credit Stress & Illustrative ECL](../04_modeling/47_credit_stress) states the three implemented
formulas and their headline caveats. This section is the "show your work" version: each implemented
formula is placed next to the published equation it comes from, every departure from that source is
listed and justified, and each derivation is checked by re-running the actual functions in
`src/models/credit_stress.py` on a worked example rather than by hand-typed arithmetic. The full
caveat text lives in `credit_stress.CREDIT_STRESS_CAVEAT` and is the canonical source of the
factual claims below; this section explains the mathematics behind it rather than restating it.

Three published sources are surgically modified here:

- Garvin, Kurian, Major & Norman (2022), RBA Research Discussion Paper 2022-03 ("Garvin et al."),
  for the stressed-PD equations (Part A) and, incidentally, the credit-loss multiplication structure
  echoed in Part B.
- KPMG in India (2025), *Expected Credit Loss (ECL): Turning Theory into Action*, for the ECL
  decomposition (Part B).
- Damodaran (2002), *Discounted Cash Flow Valuation*, for the present-value logic behind Part C.

```{admonition} How the source equations below were obtained
:class: note
Parts A and B quote directly from the RBA and KPMG PDFs (RBA RDP 2022-03 pp.7-8, 15, 39; KPMG
2025 p.5), re-extracted and checked against this project's own citations while writing this section.
Part C needs no separate literature surgery: once Part B's multi-period sum is collapsed to a single
12-month term, KPMG's own discount factor reduces to the plain one-year discount identity used
there, matching the implemented `(1 + discount_rate)` denominator in `src/models/credit_stress.py`.
(An earlier mid-year `**0.5` variant was tried and reverted before this section was written; what's
described below is the current, single-year version.) Everything else below is a direct source quote
or a direct read of `src/models/credit_stress.py`.
```

In [1]:
import sys, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for _ in range(4):
    if (PROJECT_ROOT / "reports").is_dir() and (PROJECT_ROOT / "data").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import pandas as pd

from src.models import credit_stress as cs
from src.models import svar

pd_base_assumptions = cs.load_pd_base_assumptions().set_index("segment")
lgd_ead_assumptions = cs.load_lgd_ead_assumptions().set_index("segment")
discount_rate_assumptions = cs.load_discount_rate_assumptions().set_index("segment")

assumptions = pd_base_assumptions[["pd_base"]].join(
    lgd_ead_assumptions[["lgd", "ead_aud_m"]]
).join(discount_rate_assumptions[["discount_rate"]])

# The real, SVAR-simulated cumulative-unemployment-change scenario deltas that the
# live GET /credit-risk/stress-test endpoint uses (api.main.credit_risk_stress_test
# calls this exact function, at its default horizon of 4 quarters) -- not
# illustrative stand-ins. Every worked example below uses these same three numbers,
# so the arithmetic reproduces the actual reported ECL figures, not toy ones.
scenario_names = list(cs.SCENARIO_QUANTILES)
quantiles = [cs.SCENARIO_QUANTILES[name] for name in scenario_names]
deltas, forecast_origin = svar.forecast_cumulative_unemployment_change_quantiles(quantiles=quantiles)
scenario_deltas = dict(zip(scenario_names, deltas))

print(f"Forecast origin: {forecast_origin}")
print(f"Reported scenario deltas (cumulative pp change in unemployment): {scenario_deltas}")
assumptions

Forecast origin: 2025Q4
Reported scenario deltas (cumulative pp change in unemployment): {'downside': 0.9821128128568666, 'base': 0.19959140406349096, 'upside': -0.5064202413504633}


,pd_base,lgd,ead_aud_m,discount_rate
segment,,,,
personal_loans,0.0878,0.73,1663,0.0886
mortgages,0.0207,0.16,429996,0.0680


### Part A: Stressing the probability of default

#### The source equations (Garvin et al.)

**Original, mortgages (Garvin et al. 2022, Equation 1, p.7):**

$$
PD_{ikt} \;=\; \max\Big\{\Big(\frac{PD_i^{0}}{4} + \beta_{Mort,UR}\cdot \Delta UR_t\Big)\cdot LM_k,\ \ \overline{PD}\,\Big\}
$$

where $PD_i^0$ is bank $i$'s initial (annualised) PD on its whole mortgage portfolio, $\Delta UR_t$ is the cumulative rise in
the unemployment rate since the scenario's start, $\beta_{Mort,UR}$ is a portfolio-wide sensitivity
coefficient, $LM_k$ is a vector of per-LVR-bucket multipliers that scales the whole bracket (buckets specified in 1 percentage-point
increments from 1% to 250%, calibrated by fitting a quadratic to Bergmann (2020)'s regional estimates),
and $\overline{PD}$ is a floor the paper calls a "base level", reflecting "the natural level of default that occurs even
in a strong economy due to marital failure, illness, disability and other non-cyclical risk factors"
(p.7). This floor is its own parameter, separate from $PD_i^0$. Note the division by 4: Equation 1 works with a quarterly PD, while the paper's appendix says its PDs are otherwise "annualised (or more specifically 4x the quarterly value)" (p.29), and the paper doesn't spell out the units of $\beta_{Mort,UR}$. The paper calibrates $\beta_{Mort,UR}=0.6$ (p.8) as a
value "in between" Bergmann (2020)'s regional-dispersion estimate of about 0.2 and a
cross-country regression of non-performing loans on unemployment that produced coefficients of up to
1.0. The authors call 0.6 "conservative" and note it also results in a *persistently* elevated PD (not
just a temporary spike) whenever unemployment sits above its starting level.

**Original, other portfolios incl. personal loans / credit cards (Garvin et al. 2022, Equation A45 and
Table A2, p.39):**

$$
PD_{ijt} \;=\; \max\Big\{\,PD_{ij}^{0},\ \ PD_{ij}^{0} + \beta^{GDP}_{PD}\,\Delta GDP_t + \beta^{UR}_{PD}\,\Delta UR_t +
\beta^{CRE}_{PD}\,\Delta CRE_t\,\Big\}
$$

Table A2 sets $\beta^{GDP}_{PD}=\beta^{CRE}_{PD}=0$ and $\beta^{UR}_{PD}=0.4$ for both the `CreditCards`
and `DomOthPersonal` asset classes, so for personal lending this collapses exactly to

$$
PD_{ijt} \;=\; \max\big\{\,PD_{ij}^{0},\ \ PD_{ij}^{0} + 0.4\cdot \Delta UR_t\,\big\}.
$$

Here the floor is the starting PD $PD_{ij}^{0}$ itself, not a separate natural-default parameter like Equation 1's $\overline{PD}$, and there is no division by 4. Section 3.3 (p.15) states the same 0.4-percentage-point sensitivity in prose: "a 1 percentage point
higher unemployment rate leading to a 0.4 percentage point rise in the PD for personal loans."

#### Part A: getting from the source to the code, one step at a time

The two source equations differ in three ways: only Equation 1 has the $LM_k$ multiplier, only Equation 1 divides the starting PD by 4, and only Equation 1 has a separate floor $\overline{PD}$ (Equation A45 floors at the starting PD). So personal loans are already close to the code, while mortgages need the steps marked "mortgages only" below. The chain starts from the mortgage form (writing $\beta$ for $\beta_{Mort,UR}$):

$$
PD_t \;=\; \max\big\{\big(PD^{0}/4 + \beta\cdot \Delta UR_t\big)\cdot LM_k,\ \ \overline{PD}\,\big\}
$$

**Step 1 (mortgages only): no loan-level LVR data, so drop the tiering.** Set $LM_k \equiv 1$ for every bucket, which treats every loan as the whole-book average. Because $LM_k$ multiplies the whole bracket, this removes it from the starting-PD term as well. (Personal loans never had an $LM_k$ term.)

$$
PD_t \;=\; \max\big\{\,PD^{0}/4 + \beta\cdot \Delta UR_t,\ \ \overline{PD}\,\big\}
$$

**Step 1b (mortgages only): use the annualised PD.** Equation 1 adds the unemployment increment to a quarterly PD ($PD^{0}/4$). The code treats every PD as an annual figure (`_stressed_pd` returns a decimal annual PD) and the ECL covers a full 12 months, so the `/4` is dropped (see point 6 below):

$$
PD_t \;=\; \max\big(\,PD^{0} + \beta\cdot \Delta UR_t,\ \ \overline{PD}\,\big)
$$

**Step 2: name the starting PD, and swap the mortgage floor.** The starting PD is the disclosed baseline for both segments, so $PD^{0}=PD_{base}$. Mortgages have a separate floor $\overline{PD}$ and this project has no estimate of it, so it is set to $PD_{base}$ too. Personal loans join the chain here: Equation A45, with Table A2's zero GDP/CRE coefficients (so $\beta=\beta^{UR}_{PD}=0.4$), already floors at the starting PD. Both segments now have the same form:

$$
PD_t \;=\; \max\big(\,PD_{base} + \beta\cdot \Delta UR_t,\ \ PD_{base}\,\big)
$$

**Step 3: factor $PD_{base}$ out of the max.** Using the identity $\max(a+b,\,a)=a+\max(b,0)$ with
$a=PD_{base}$ and $b=\beta\cdot\Delta UR_t$:

$$
PD_t \;=\; PD_{base} + \max\big(0,\ \ \beta\cdot \Delta UR_t\big)
$$

**Step 4: match units.** $\beta$ and $\Delta UR_t$ are both stated in the paper in percentage points
(e.g. "0.6", "a 1 percentage point rise"), but `pd_base`/`pd_stressed` are stored in the code as decimal
fractions. Renaming to the code's own symbols ($\beta\to s$, $\Delta UR_t\to \Delta u_{cum}$) and
dividing the sensitivity term by 100:

$$
PD_t \;=\; PD_{base} + \max\Big(0,\ \ s\cdot \Delta u_{cum}/100\Big)
$$

**Step 5: add the project's own safety clip, and undo Step 3's factoring.** Cap the result at 100%
(not present in either source equation), then run the Step 3 identity in reverse to land on the form the
code uses:

$$
PD_{stressed} \;=\; \min\!\Big(1,\ \ PD_{base} + \max\big(0,\ s\cdot \Delta u_{cum}/100\big)\Big)
\;=\;\min\!\Big(1,\ \max\big(PD_{base},\ \ PD_{base} + s\cdot \Delta u_{cum}/100\big)\Big)
$$

which is exactly `min(1.0, max(pd_base, pd_base + effect))` in `_stressed_pd`.

#### Part A: every departure from the source

1. **Mortgage LVR-bucket multiplier $LM_k$ dropped** (Step 1). Equation 1 is loan-level and LVR-tiered, and $LM_k$
   scales the whole bracket. With no loan-level LVR data, this project applies no LVR scaling (one
   portfolio-average treatment). It's the biggest single simplification in Part A, and it only affects
   mortgages: personal loans never had an LVR term in the source (unsecured lending).
2. **Mortgage floor $\overline{PD}$ replaced with $PD_{base}$** (Step 2). Equation 1's floor is a separate
   "natural default rate" that the project has no disclosed estimate for, so the segment's own starting PD
   is substituted. Net effect: stressed PD can never fall *below* its starting value, even under the
   "upside" scenario where $\Delta u_{cum}$ can be negative, whereas in Equation 1 it could glide down toward
   $\overline{PD}$. **Personal loans have no floor departure:** Equation A45 already floors at the starting
   PD, exactly as the code does.
3. **Upper clip at 1.0 added** (Step 5). Not present in Equation 1 or Equation A45; a project-added
   bound so `pd_stressed` is always a valid probability.
4. **The `/100` conversion is a units bridge, not a modelling change** (Step 4). `s` is stored as
   "percentage points of PD per percentage point of unemployment change" (matching the paper's own
   "0.4pp" / "0.6" language), while `pd_base`/`pd_stressed` are decimal fractions, which is also the
   only scale on which the `min(1.0, ...)` clip is meaningful.
5. **Two source equations unified into one shared function.** For personal loans this is a very close
   match (Table A2 already zeroes the GDP/CRE terms, and A45's floor and units already fit the code). For
   mortgages it is a larger simplification, since the LVR-tiering mechanism (point 1) is dropped and
   points 2 and 6 also apply.
6. **The `/4` is not applied to mortgages: an open simplification** (Step 1b). Equation 1 adds the
   increment to a quarterly PD ($PD_i^0/4$); the code adds it to the annual PD. If the paper means the
   increment as a quarterly amount, the annualised effect would be about four times as large and the code
   understates the mortgage stress (downside case below:
   $2.07\% + 4\times 0.5893\% \approx 4.43\%$ instead of 2.66%). The paper doesn't state the units of
   $\beta_{Mort,UR}$, so this page can't settle which reading is intended. The project uses the annualised
   one, which matches Equation A45 for personal loans, so the mortgage ECL should be read as illustrative
   in that light too.

#### Worked example: mortgages at the real downside delta

The reported "downside" scenario delta is $\Delta u_{cum}=0.9821$ percentage points: the actual SVAR-simulated, horizon-4 cumulative unemployment change computed in the setup cell above, the same number behind the [Key
Findings](../00_results_at_a_glance) page and [§4.7](../04_modeling/47_credit_stress), not an
illustrative stand-in. Step by step, for mortgages:

| Step | Expression | Value |
|---|---|---|
| Look up | $PD_{base}$ | 2.07% |
| Look up | $s=\beta_{Mort,UR}$ | 0.6 |
| Multiply (still pp) | $s\cdot \Delta u_{cum} = 0.6\times 0.9821$ | 0.5893 |
| Convert to decimal | $s\cdot \Delta u_{cum}/100 = 0.5893/100$ | 0.5893% |
| Add | $PD_{base}+0.5893\% = 2.07\%+0.5893\%$ | 2.6593% |
| Clip at 100% | $\min(1,\ 2.6593\%)$ | **2.66% = $PD_{stressed}$** |

The code cell below repeats this arithmetic for both segments (mortgages and personal loans) at the
same real downside delta, using the real assumption files and the real
`mortgage_stressed_pd`/`personal_loan_stressed_pd` functions, and checks that the hand and coded values
agree to within $10^{-12}$, not just "by eye" as in the table above.

In [2]:
DELTA_UR_DOWNSIDE = scenario_deltas["downside"]  # real SVAR-simulated value, not illustrative


def hand_stressed_pd(pd_base: float, s: float, delta_ur: float) -> float:
    effect = s * delta_ur / 100
    return min(1.0, max(pd_base, pd_base + effect))


rows = []
for segment, s, fn in [
    ("mortgages", cs.MORTGAGE_UR_SENSITIVITY, cs.mortgage_stressed_pd),
    ("personal_loans", cs.PERSONAL_LOAN_UR_SENSITIVITY, cs.personal_loan_stressed_pd),
]:
    pd_base = float(pd_base_assumptions.loc[segment, "pd_base"])
    hand = hand_stressed_pd(pd_base, s, DELTA_UR_DOWNSIDE)
    coded = fn(pd_base, DELTA_UR_DOWNSIDE)
    assert abs(hand - coded) < 1e-12, f"{segment}: hand and coded PD_stressed disagree"
    rows.append(
        {
            "segment": segment,
            "PD_base": pd_base,
            "s (pp of PD per pp of UR)": s,
            "delta_u_cum (pp, real downside scenario)": DELTA_UR_DOWNSIDE,
            "PD_stressed (hand)": hand,
            "PD_stressed (credit_stress.py)": coded,
        }
    )

pd.DataFrame(rows).style.format(
    {
        "PD_base": "{:.4%}",
        "PD_stressed (hand)": "{:.4%}",
        "PD_stressed (credit_stress.py)": "{:.4%}",
    }
)

,segment,PD_base,s (pp of PD per pp of UR),"delta_u_cum (pp, real downside scenario)",PD_stressed (hand),PD_stressed (credit_stress.py)
0,mortgages,2.0700%,0.600000,0.982113,2.6593%,2.6593%
1,personal_loans,8.7800%,0.400000,0.982113,9.1728%,9.1728%


### Part B: From default probability to a dollar loss

#### The source equation (KPMG)

**Original (KPMG in India, 2025, p.5):**

$$
ECL \;=\; \sum_{t=1}^{T} PD_t \times LGD_t \times EAD_t \times D_t
$$

"$D$ is discount rate which can be computed as Effective Interest Rate (EIR) under GMM and Simplified
approach or Credit-Adjusted Effective Interest Rate (CEIR) under POCI approach" (p.5). This is a general
*lifetime* ($T$-period) formulation: IFRS 9's General Measurement Model requires either a 12-month or a
lifetime ECL depending on which "stage" an exposure sits in.

In plain terms: **Stage 1** is a performing exposure with no significant increase in credit risk (SICR)
since origination, and gets 12 months of expected loss. **Stage 2** has a SICR but isn't in default yet,
and **Stage 3** is credit-impaired or in default; both get the full lifetime sum above instead of a
single 12-month term. GMM is that same three-stage General Measurement Model, the standard IFRS 9
approach this project's one Stage 1 term is a slice of. POCI is a separate track for loans that were
already credit-impaired when a bank acquired them, which doesn't apply to anything in this
project.

**The same multiplicative structure already exists inside Garvin et al. itself**, one layer down from
Part A, in the non-mortgage write-off calculation (Equations A44 and A51, both on p.39; the paper applies
A44 to every non-mortgage loan class except "overseas and other loans"):

$$
LR_{ijt} = PD_{ijt}\times LGD_{ijt} \qquad\qquad \text{Write-off}_{ijt} = LR_{ijt}\times AssetBal_{ij,t-1}
$$

Combined, that is $\text{Write-off}_{ijt} = PD_{ijt}\times LGD_{ijt}\times AssetBal_{ij,t-1}$: the same
three-factor multiplicative form as KPMG's ECL, just undiscounted and framed as a stress-test write-off
rather than an accounting provision. So the implemented ECL formula below grafts KPMG's
accounting frame (discounted, 12-month, Stage-1) onto Garvin et al.'s own credit-loss arithmetic
(PD $\times$ LGD $\times$ exposure), with $EAD$ standing in for $AssetBal_{t-1}$.

#### Part B: getting from the source to the code, one step at a time

**Step 1: collapse the lifetime sum to a single 12-month term.** This project only computes a
Stage-1, 12-month ECL (no SICR/staging logic exists to put any exposure into Stage 2 or 3), so the sum
over $t=1,\dots,T$ has exactly one term, $t=1$:

$$
ECL \;=\; PD_1 \times LGD_1 \times EAD_1 \times D_1
$$

**Step 2: drop the now-redundant time subscript.** With only one period left, the $t=1$ label carries
no information, so it is dropped from every symbol:

$$
ECL \;=\; PD \times LGD \times EAD \times D
$$

**Step 3: substitute in what $PD$ and $D$ actually are.** $PD$ is Part A's stressed probability,
$PD_{stressed}$; $D$ is specialised in Part C to the plain one-year discount factor $\tfrac{1}{1+r}$:

$$
ECL \;=\; PD_{stressed} \times LGD \times EAD \times \frac{1}{1+r}
\;=\; \frac{PD_{stressed}\times LGD \times EAD}{1+r}
$$

which is exactly `run_credit_stress_test`'s `stressed_pd * lgd * ead / (1 + discount_rate)`.

#### Part B: every departure from the source

1. **KPMG's $\sum_{t=1}^{T}$ collapsed to a single $T=1$ term** (Step 1). This is explicitly a 12-month,
   Stage-1-only ECL: with no loan-level origination-vs-current credit-risk or delinquency data to detect
   SICR, the lifetime terms a real Stage 2/3 exposure would add to the sum are simply absent, not
   estimated as zero.
2. **$EAD$ is a portfolio-level aggregate, not a per-loan or per-facility exposure.** NAB's disclosed
   post-CCF/post-CRM Table CR6 figure is used directly as $EAD$ for the whole exposure class, so the
   resulting $ECL_{aud\,m}$ is an aggregate dollar figure for that class, not a granular provision build
   the way a real bank's ECL engine would compute it loan by loan. (Post-CCF means after applying the
   Credit Conversion Factor, which turns an undrawn credit line into an equivalent drawn amount;
   post-CRM means after Credit Risk Mitigation, i.e. after collateral and guarantees are accounted for.
   NAB's Table CR6 figure already has both applied, so it's the fully Basel-adjusted exposure, not a
   raw limit.)
3. **$D_t$ is specialised to a single $(1+r)^{-1}$ factor** (Step 3, derived in full in Part C) rather
   than left as a generic, possibly time-varying EIR-based discount factor across
   $t=1,\dots,T$ periods, a direct consequence of collapsing to $T=1$ in Step 1.

#### Worked example: continuing the real downside case

Step by step, continuing from Part A ($PD_{stressed}=2.6593\%$):

| Step | Expression | Value |
|---|---|---|
| Look up | $LGD$ | 16% |
| Look up | $EAD$ | \$429,996m |
| Multiply | $PD_{stressed}\times LGD\times EAD = 2.6593\%\times16\%\times\$429{,}996m$ | \$1,829.56m |
| Look up | $r$ | 6.80% |
| Add 1 | $1+r = 1+0.0680$ | 1.0680 |
| Divide | $\$1{,}829.56m \div 1.0680$ | **\$1,713.07m = ECL** |

This \$1,713.07m is the real "downside" row for mortgages in `reports/tableau/credit_stress.csv`
and it feeds the [§4.7](../04_modeling/47_credit_stress) chart; it is not a rounded illustrative number. The code cell below re-derives it from the
actual `run_credit_stress_test` output (not a hand-typed constant) and checks the two agree to within
$10^{-9}$:

In [3]:
def hand_ecl(pd_stressed: float, lgd: float, ead: float, r: float) -> float:
    return pd_stressed * lgd * ead / (1 + r)


segment = "mortgages"
pd_base = float(pd_base_assumptions.loc[segment, "pd_base"])
lgd = float(lgd_ead_assumptions.loc[segment, "lgd"])
ead = float(lgd_ead_assumptions.loc[segment, "ead_aud_m"])
r = float(discount_rate_assumptions.loc[segment, "discount_rate"])

pd_stressed = cs.mortgage_stressed_pd(pd_base, DELTA_UR_DOWNSIDE)
hand = hand_ecl(pd_stressed, lgd, ead, r)

coded = cs.run_credit_stress_test(
    {"downside": DELTA_UR_DOWNSIDE},
    scenario_weights={"downside": 1.0},
)
coded_value = float(
    coded.loc[(coded["segment"] == segment) & (coded["scenario"] == "downside"), "ecl_aud_m"].iloc[0]
)

assert abs(hand - coded_value) < 1e-9, "hand and coded ECL disagree"

print(
    f"Hand:  ECL = PD_stressed({pd_stressed:.4%}) x LGD({lgd:.0%}) x EAD(${ead:,.0f}m) "
    f"/ (1+{r:.2%}) = ${hand:,.2f}m"
)
print(f"Code:  run_credit_stress_test(...) -> ${coded_value:,.2f}m")
print(f"Match: {abs(hand - coded_value) < 1e-9}")

Hand:  ECL = PD_stressed(2.6593%) x LGD(16%) x EAD($429,996m) / (1+6.80%) = $1,713.07m
Code:  run_credit_stress_test(...) -> $1,713.07m
Match: True


### Part C: Discounting the 12-month shortfall

Part B already specialised KPMG's discount factor $D_t$ down to a single $t=1$ term, $D$. What's left
to pin down is what that one-period factor actually *is*, and why dividing by $(1+r)$ is the right
thing to do, not just an assertion.

#### Deriving $PV = \dfrac{CF}{1+r}$ from first principles

Suppose you have \$1 today and can invest it at rate $r$ for one year. After a year it has grown to

$$
1 \times (1+r)
$$

More generally, an amount $PV$ invested today grows to $PV\times(1+r)$ after one year. Now flip the
question around: if a cash flow $CF$ is going to land in exactly one year, what amount $PV$, invested
today at the same rate $r$, would grow into exactly that $CF$? Set the two equal and solve for $PV$:

$$
PV\times(1+r) \;=\; CF
\qquad\Longrightarrow\qquad
PV \;=\; \frac{CF}{1+r}
$$

That is the entire derivation: $PV=\dfrac{CF}{1+r}$ is just "grows to $CF$" run backwards. This is the
general present-value identity behind discounted cash flow valuation (Damodaran 2002).

**Implemented:** exactly this identity, with $CF$ = the 12-month expected credit-loss shortfall
$PD_{stressed}\times LGD\times EAD$ (the undiscounted number computed in Part B) and $r$ = the segment's
RBA lending-rate proxy for its effective interest rate (6.80% for mortgages, RBA Table F5; 8.86% for
personal loans, RBA Table F8):

$$
ECL \;=\; \frac{PD_{stressed}\times LGD\times EAD}{1+r}
$$

**Departure from the source:** essentially none, once Part B's $T=1$ collapse is taken as given. Of the
three formulas in this appendix, this is the one with the least surgery: Part A drops a mortgage
LVR-tiering mechanism, substitutes a mortgage floor and sets aside the mortgage `/4`; Part B collapses a lifetime sum and relabels a portfolio
exposure; Part C is just the plain one-year discount factor implied by treating the 12-month shortfall
as a single cash flow. An earlier version of this formula *did* apply a mid-year, $(1+r)^{-0.5}$
adjustment by analogy to general DCF practice; it was dropped because neither AASB 9 nor the KPMG note
prescribes a within-year timing convention for ECL, and the plain factor doesn't need one.

No separate worked check is needed for Part C: the $(1+r)$ term is already exercised inside the Part B
check above (`hand_ecl` divides by `(1 + r)` explicitly, and matched the coded function to `1e-9`).

### Part D: Combining scenarios

$$
ECL_{weighted} \;=\; \sum_{s\,\in\,\{downside,\,base,\,upside\}} w_s \cdot ECL_s
$$

$w_s$ is NAB's own disclosed FY2025 Annual Report macroeconomic scenario probability weighting (base
55%, downside 42.5%, upside 2.5%). This is not a modification of a published equation the way Parts A-C
are (it is a straightforward probability-weighted sum), but it is worth stating precisely because of
what it grafts together: **real NAB weights, applied to this project's own generic (non-NAB) PD-stress
mechanism from Parts A-C, not to NAB's own ECL model.** The weights are genuine and disclosed; the thing
they are weighting is this project's construction, not NAB's.

**Worked, step by step, for mortgages, using the real SVAR-simulated scenario deltas from the setup
cell**, the same three numbers the live report is built from, so this reproduces the actual reported
figure rather than an illustrative total:

| Scenario $s$ | $w_s$ | $\Delta u_{cum}$ (pp, real) | $PD_{stressed}$ | $ECL_s$ (\$m) | $w_s\times ECL_s$ (\$m) |
|---|---|---|---|---|---|
| downside | 0.425 | +0.9821 | 2.66% | 1,713.07 | 728.05 |
| base | 0.550 | +0.1996 | 2.19% | 1,410.62 | 775.84 |
| upside | 0.025 | -0.5064 | 2.07% | 1,333.47 | 33.34 |
| **Sum** | | | | | **1,537.23 = $ECL_{weighted}$** |

Reading the downside row: $\Delta u_{cum}=0.9821$pp is exactly Part A's worked example, giving
$PD_{stressed}=2.6593\%$, which feeds Part B/C's $\dfrac{2.6593\%\times16\%\times\$429{,}996m}{1.068}=
\$1{,}713.07m$, the same number derived there, reused rather than recomputed. Multiplying by that
scenario's weight, $0.425\times\$1{,}713.07m=\$728.05m$, and summing all three weighted rows gives
\$1,537.23m: **the exact mortgages figure reported on the [Key Findings](../00_results_at_a_glance)
page and in [§4.7](../04_modeling/47_credit_stress)**, not a rounded coincidence.

The code cell below re-derives every number in that table from `run_credit_stress_test`, using the real
scenario deltas from the setup cell (for both segments, not just mortgages), so nothing above is a
hand-typed constant. Its output should match `reports/tableau/credit_stress.csv` exactly:

In [4]:
# The real scenario deltas computed in the setup cell above (the same values the
# live GET /credit-risk/stress-test endpoint uses), not illustrative ones.
full = cs.run_credit_stress_test(scenario_deltas)  # uses the real cs.SCENARIO_PROBABILITY_WEIGHTS

weighted = full.groupby("segment").apply(
    lambda g: float((g["ecl_aud_m"] * g["probability_weight"]).sum())
)

print(f"Scenario weights used: {cs.SCENARIO_PROBABILITY_WEIGHTS}")
print(f"Scenario deltas used (real, from the setup cell): {scenario_deltas}")
for segment in cs.SEGMENT_ORDER:
    print(f"{segment}: ECL_weighted = ${weighted[segment]:,.2f}m")

full[["segment", "scenario", "probability_weight", "delta_unemployment_cumulative", "pd_stressed", "ecl_aud_m"]]

Scenario weights used: {'downside': 0.425, 'base': 0.55, 'upside': 0.025}
Scenario deltas used (real, from the setup cell): {'downside': 0.9821128128568666, 'base': 0.19959140406349096, 'upside': -0.5064202413504633}
personal_loans: ECL_weighted = $100.26m
mortgages: ECL_weighted = $1,537.23m


,segment,scenario,probability_weight,delta_unemployment_cumulative,pd_stressed,ecl_aud_m
0,personal_loans,downside,0.425,0.982113,0.091728,102.294160
1,personal_loans,base,0.550,0.199591,0.088598,98.803537
2,personal_loans,upside,0.025,-0.506420,0.087800,97.913211
3,mortgages,downside,0.425,0.982113,0.026593,1713.070365
4,mortgages,base,0.550,0.199591,0.021898,1410.615466
5,mortgages,upside,0.025,-0.506420,0.020700,1333.470742


### Summary: every modification in one table

| # | Equation | Original | Implemented | Why |
|---|----------|----------|-------------|-----|
| 1 | Stressed PD (mortgages) | Per-LVR-bucket multiplier $LM_k$ on the whole bracket (Eq. 1) | $LM_k \equiv 1$ | No loan-level LVR data in this project |
| 2 | Stressed PD (mortgages) | Floor $\overline{PD}$ = separate natural-default rate (Eq. 1); Eq. A45 already floors at the starting PD | Floor = $PD_{base}$ | No disclosed estimate of $\overline{PD}$ exists. No departure for personal loans |
| 3 | Stressed PD (both) | No upper bound in Eq. 1 / Eq. A45 | Clipped at $PD_{stressed}\le 1$ | Numerical safety bound |
| 4 | Stressed PD (both) | Percentage-point arithmetic throughout | Explicit `/100` conversion | `pd_base` is stored as a decimal fraction |
| 5 | Stressed PD (both) | Two separate equations (mortgages vs. other portfolios) | One shared function | Personal loans: near-exact match. Mortgages: LVR mechanism dropped |
| 6 | Stressed PD (mortgages) | Starting PD divided by 4, so the increment is added to a quarterly PD (Eq. 1) | Annual PD used directly | Code treats every PD as annual; the paper doesn't state the units of $\beta_{Mort,UR}$, so this is an open simplification |
| 7 | ECL structure | $\sum_{t=1}^{T}$ (lifetime, all stages) | Single $T=1$ term | 12-month, Stage-1-only; no SICR/staging data available |
| 8 | ECL structure | $EAD_t$ = per-facility exposure at default | $EAD$ = disclosed portfolio aggregate | Only a portfolio-level Pillar 3 figure is available |
| 9 | Discounting | $D_t$: generic multi-period EIR discount factor | $(1+r)^{-1}$, one full year, no within-year timing adjustment | Direct consequence of collapsing to $T=1$; AASB 9 requires EIR discounting but prescribes no within-year timing convention |
| 10 | Scenario combination | -- (no published equation being modified) | NAB's real scenario weights $\times$ this project's generic mechanism | Real weights, but not weighting NAB's own model |

#### References

Damodaran, A. (2002). *Discounted cash flow valuation* [NYU Stern equity valuation course notes].
https://pages.stern.nyu.edu/~adamodar/pdfiles/eqnotes/dcfallOld.pdf

Garvin, N., Kurian, S., Major, M., & Norman, D. (2022). *Macrofinancial stress testing on Australian
banks* (Research Discussion Paper No. RDP 2022-03). Reserve Bank of Australia.
https://www.rba.gov.au/publications/rdp/2022/2022-03.html

KPMG in India. (2025). *Expected credit loss (ECL): Turning theory into action*.
https://assets.kpmg.com/content/dam/kpmgsites/in/pdf/2025/01/expected-credit-loss-ecl.pdf

National Australia Bank. (2025a). *Pillar 3 disclosure report -- 30 September 2025* (Table CR6, Credit
Risk Exposures).

National Australia Bank. (2025b). *Annual report 2025* (Note 17, Provision for Credit Impairment;
macroeconomic scenario probability weightings).
